# D435：异步读取彩色与深度图像

通过 `D435()` 和 `Runtime` 直接在 Notebook 的事件循环中使用相机。两个流共享一个后台采集进程；上层只接触 `Frame`、`Image` 和 NumPy 数组。

运行前选择能导入 RSim、ROS 2 和 `scipykit` 的 kernel，并安装对应 ROS 发行版的 `realsense2_camera` 驱动。设备配置集中在下方，不需要手动启动 ROS 节点。图片、原始数组和采集报告保存到项目根目录 `assets/d435/`。

In [1]:
%load_ext autoreload
%autoreload 3 --print --log

In [2]:
import asyncio
import json
import os
from pathlib import Path

import numpy as np
from rsim import Runtime
from rsim.drivers import D435

# 支持从项目根目录或 examples/ 启动，不绑定本机路径。
ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "pyproject.toml").exists() and (p / "rsim").is_dir()
    )
ASSETS = ROOT / "assets" / "d435"
ASSETS.mkdir(parents=True, exist_ok=True)
os.environ["NOTEBOOK_ASSETS_ROOT"] = os.path.relpath(
    ROOT / "assets", Path.cwd()
    )
os.environ["NOTEBOOK_NAME"] = "d435"

from scipykit.mtp_initializer import subplots, disp, plt, show_image

## 配置

示例使用两路 640×480、15 Hz。`depth_profile` / `color_profile` 格式为 `宽x高x帧率`，须选择设备在当前 USB 连接下支持的模式；可以在此改为更高分辨率，或通过环境变量覆盖。帧率是请求值，下面会统计实际收到的新帧。

空序列号选择可用的 D435；连接多台时应填写目标序列号。所有复用同一相机的调用者应使用相同的序列号写法、两路 profile 和 history。修改配置前先关闭已有使用者。

若取流超时，查看 `driver.log`，检查 USB 连接和支持的模式；可尝试降低帧率，例如深度 `480x270x6`、彩色 `424x240x6`；优先确认使用 USB 3 数据线与端口。

In [ ]:
CONFIG = {
    "serial": os.environ.get("RSIM_D435_SERIAL", ""),
    "depth_profile": os.environ.get("RSIM_D435_DEPTH_PROFILE", "640x480x15"),
    "color_profile": os.environ.get("RSIM_D435_COLOR_PROFILE", "640x480x15"),
    "history": 16,
    "log_path": ASSETS / "driver.log",
}
SAMPLE_COUNT = 90
color = D435(stream="color", **CONFIG)
depth = D435(stream="depth", **CONFIG)

## 取帧、按时间戳回查和释放设备

`get()` 返回最新帧，`get(after=frame.sequence)` 等待新图像；历史查询必须同时传入源时间戳和时间域。两路读取得到的是各自最新帧，**没有承诺曝光同步或像素对齐**。

下面用 `async with` 包住整个采集过程，异常或中断也会执行资源清理。最后一个使用者退出后，后台相机驱动会关闭；已取出的只读数组仍可用于后续显示。

In [ ]:
stamps = {"color": [], "depth": []}
async with Runtime(color, depth):
    rgb_frame, depth_frame = await asyncio.gather(
        color.image.get(timeout=30), depth.image.get(timeout=30))

    for _ in range(SAMPLE_COUNT):
        rgb_frame, depth_frame = await asyncio.gather(
            color.image.get(after=rgb_frame.sequence, timeout=5),
            depth.image.get(after=depth_frame.sequence, timeout=5))
        stamps["color"].append(rgb_frame.stamp_ns)
        stamps["depth"].append(depth_frame.stamp_ns)

    recovered = await depth.image.get(
        timestamp_ns=depth_frame.stamp_ns, clock=depth_frame.clock
        )
    assert recovered is depth_frame
    assert color.source.producer.worker_pid == depth.source.producer.worker_pid
    source_pid = color.source.producer.worker_pid

rgb = rgb_frame.data.pixels
depth_mm = depth_frame.data.pixels
assert rgb_frame.data.encoding == "rgb8" and rgb.dtype == np.uint8
assert depth_frame.data.encoding == "16UC1" and depth_mm.dtype == np.uint16
assert all(
    isinstance(a, np.memmap) and not a.flags.writeable
    for a in (rgb, depth_mm)
    )
# 驱动可能在模式不支持时退回默认值，因此核对实际尺寸。
for name, pixels in (("color", rgb), ("depth", depth_mm)):
    width, height, _ = map(int, CONFIG[f"{name}_profile"].split("x"))
    assert pixels.shape[:2] == (height, width), f"{name} profile 未生效"
    assert len(set(stamps[name])) == SAMPLE_COUNT, f"{name} 重复了源帧"

summary = {}
for name, frame in (("color", rgb_frame), ("depth", depth_frame)):
    series = np.array(stamps[name], dtype=np.int64)
    summary[name] = {
        "shape":
            list(frame.data.pixels.shape),
        "encoding":
            frame.data.encoding,
        "frames":
            len(series),
        "observed_hz":
            round((len(series) - 1) * 1e9 / int(series[-1] - series[0]),
                    2),
        "max_source_gap_ms":
            round(float(np.diff(series).max()) / 1e6, 2),
        "stamp_ns":
            frame.stamp_ns,
        "clock":
            frame.clock,
        "received_ns":
            frame.received_ns,
        "readonly_memmap":
            True,
        }
summary["shared_worker_pid"] = source_pid
summary["timestamp_lookup"] = "passed"
print(json.dumps(summary, indent=2, ensure_ascii=False))

{
  "color": {
    "shape": [
      480,
      640,
      3
    ],
    "encoding": "rgb8",
    "frames": 90,
    "observed_hz": 9.96,
    "max_source_gap_ms": 667.18,
    "stamp_ns": 1790143712193725586,
    "clock": "device:d435:",
    "received_ns": 1790143712262773077,
    "readonly_memmap": true
  },
  "depth": {
    "shape": [
      480,
      640
    ],
    "encoding": "16UC1",
    "frames": 90,
    "observed_hz": 9.81,
    "max_source_gap_ms": 667.18,
    "stamp_ns": 1790143712327092529,
    "clock": "device:d435:",
    "received_ns": 1790143712338272095,
    "readonly_memmap": true
  },
  "shared_worker_pid": 475912,
  "timestamp_lookup": "passed"
}


## 彩色与深度预览

这里的深度流来自 RealSense ROS 驱动的 `16UC1` 图像，驱动[将深度单位归一为毫米](https://github.com/realsenseai/realsense-ros/blob/4.58.4/realsense2_camera/src/base_realsense_node.cpp#L253)。0 表示无有效深度；显示时转换为米并遮罩无效像素，原始数组保持不变。

彩色和深度相机的视场、分辨率和光心不同，不能直接把同一像素位置当作同一空间点；此示例尚未提供内外参或深度到彩色的配准。

In [ ]:
show_image(rgb, key="color")

valid = depth_mm > 0
depth_m = np.ma.masked_where(~valid, depth_mm.astype(np.float32) * 0.001)
upper = max(0.1, float(np.percentile(depth_m.compressed(), 98))) if valid.any() else 1.0
fig, ax = subplots(figsize=(6.66, 3.8))
im = ax.imshow(depth_m, cmap="viridis", vmin=0, vmax=upper)
ax.set_title("D435 depth (not registered to color)")
ax.set_axis_off()
fig.colorbar(im, ax=ax, label="Depth / m", shrink=0.85)
disp(fig, "depth")
plt.close(fig)
print(f"有效深度像素：{valid.mean():.1%}")

../../assets/d435/color.png

../../assets/d435/depth.png

有效深度像素：82.4%


## 帧间隔与保存

下图展示应用实际取到的源帧时间间隔，包括相机丢帧或应用跳过旧帧造成的间隔；它不是硬件曝光抖动的独立测量。`get()` 采用最新帧语义，不保证逐帧无损录像。

In [6]:
fig, ax = subplots(figsize=(6.66, 3))
for name in ("color", "depth"):
    gaps_ms = np.diff(np.array(stamps[name], dtype=np.int64)) / 1e6
    ax.plot(gaps_ms, label=name, alpha=0.8)
ax.set_xlabel("Received frame interval index")
ax.set_ylabel("Source timestamp gap / ms")
ax.legend()
disp(fig, "frame_intervals")
plt.close(fig)

np.save(ASSETS / "color.npy", rgb)
np.save(ASSETS / "depth_mm.npy", depth_mm)
report = {"config": {k: str(v) if isinstance(v, Path) else v for k, v in CONFIG.items()},
          "summary": summary, "source_stamps_ns": stamps}
(ASSETS / "capture.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print("原始数组、预览和采集报告已保存到", ASSETS.relative_to(ROOT))

../../assets/d435/frame_intervals.png

原始数组、预览和采集报告已保存到 assets/d435


需要持续处理时，将业务逻辑放进同一个 `async with Runtime(...)` 内，或用 `Map` / `ProcessSensor` 组合计算任务。这里的跨进程数组以只读共享内存映射返回；ROS 驱动到 Python 的接入路径仍可能复制数据，不代表整条链路零拷贝。